In [0]:
%pip install openpyxl


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
import re
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

RUTA_EXCEL = "/Volumes/workspace/bronce/archivos/Datos_maestros.xlsx"


# Convierte un nombre de columna en un nombre técnico válido para Delta
# Ejemplo: "Criticidad (Remedy)" -> "criticidad_remedy"
def nombre_tecnico(columna):
    texto = columna.lower()
    texto = texto.replace("á", "a").replace("é", "e").replace("í", "i")
    texto = texto.replace("ó", "o").replace("ú", "u").replace("ñ", "n")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return texto.strip("_")


# Convierte una tabla de pandas en un DataFrame de Spark con todas las columnas como texto
def a_spark(tabla_pd):
    campos = []
    for columna in tabla_pd.columns:
        campos.append(StructField(columna, StringType(), True))
    esquema = StructType(campos)
    return spark.createDataFrame(tabla_pd, schema=esquema)


In [0]:
from pyspark.sql import functions as F

# Leer con pandas, todo como texto (dtype=str)
matriz_pd = pd.read_excel(RUTA_EXCEL, sheet_name="Matriz", usecols=range(33), dtype=str)
matriz_pd = matriz_pd.dropna(how="all")
matriz_pd = matriz_pd.astype(object).where(matriz_pd.notna(), None)

# Renombrar columnas a nombres técnicos
nuevos_nombres = []
for columna in matriz_pd.columns:
    nuevos_nombres.append(nombre_tecnico(columna))
matriz_pd.columns = nuevos_nombres

# Pasar a Spark y agregar la fecha de carga
matriz = a_spark(matriz_pd)
matriz = matriz.withColumn("fecha_carga", F.current_timestamp())

# Guardar como tabla Delta
matriz.write.mode("overwrite").saveAsTable("workspace.bronce.matriz")

print("Tabla bronce.matriz guardada:", matriz.count(), "filas")


/local_disk0/.ephemeral_nfs/envs/pythonEnv-d9c72df0-d365-4b7a-bb54-9e9ca22c62f5/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Tabla bronce.matriz guardada: 460 filas


In [0]:
combos_pd = pd.read_excel(RUTA_EXCEL, sheet_name="Combos", header=3, usecols="B:I", dtype=str)
combos_pd = combos_pd.astype(object).where(combos_pd.notna(), None)

nuevos_nombres = []
for columna in combos_pd.columns:
    nuevos_nombres.append(nombre_tecnico(columna))
combos_pd.columns = nuevos_nombres

combos = a_spark(combos_pd)
combos = combos.withColumn("fecha_carga", F.current_timestamp())
combos.write.mode("overwrite").saveAsTable("workspace.bronce.combos")

print("Tabla bronce.combos guardada:", combos.count(), "filas")


Tabla bronce.combos guardada: 3 filas


In [0]:
display(spark.table("workspace.bronce.matriz"))


servicio_de_negocio,subservicio_de_negocio,administracion,administracion_central,administracion_general,codigo_del_aplicativo,clave_o_id,nombre_completo_de_la_aplicacion,acronimo_de_la_aplicacion,descripcion,objetivo,responsable_sat_rape,administrador_de_soluciones_de_negocio,responsable_funcional_de_soluciones_de_negocio,fabrica_responsable,lider_de_la_fabrica,criticidad_remedy,severidad_remedy,aplicacion_critica,tipo_de_arquitectura,lenguaje_de_desarrollo,manejador_y_version_de_la_base_de_datos,nombre_de_la_base_de_datos_productiva,sistema_operativo_front_end,sistema_operativo_middleware,sistema_operativo_para_el_manejador_de_la_base_de_datos,pistas_auditoria,descripcion_pistas_de_auditoria,tipo_movilidad,aplicacion_en_la_nube,aplicacion_o_componente,mesa_de_servicio,tipo_de_autenticacion,fecha_carga
Identificación del Contribuyente,"Procesos de Apoyo, Actualización y Extracción",APE3,José Omar Mancilla Gómez,AGSC,1,1,Actualización de DARIO ALR´S Sedes,DARIO ALR's,Estandarizar información entre la base de datos de DARIO Central y las Regionales de Recaudación.,Estandarizar información entre la base de datos de DARIO Central y las Regionales de Recaudación.,Araceli Gándara Bautista,Everardo Sánchez Hernández,Christian Alejandro Nolasco Palacios,Fábrica SDMA 7 DS,null,Baja,3,No Crítica,Legada,Informix 4GL 10 Visual Basic 6.0 Tuxedo 8,Informix 10,DARIO1 ALR##_LN,UNIX HP-UX 11.11,Unix 11.11,UNIX HP-UX 11.11,SI,Usuario y fecha de modificación al nivel tramite,No Aplica,NO,Aplicación,SI,Omniguard,2026-10-20T22:48:09.176Z
Fiscalización,Ejecutar Actos de Fiscalización,APE2,Alejandro Federico Tejeda Muñoz,AGAFF,2,2,Sistema Único de Información Integral,SUII,Herramienta interna que permite controlar el estatus de propuestas y órdenes de fiscalización.,Administrar las cuentas que tienen acceso a las consultas siguientes a través de intranet: - Transacciones - Devoluciones (Automáticas y Manuales) - R1 y R2,Raúl Quiroz Sosa,Ingrid Bernal Campuzano,Santa Isabel Villanueva Hernández,Fábrica SDMA 7 DS,null,Media,3,No Crítica,Legada,Visual Studio 6.0,Microsoft SQL Server 2008,Central DinámicaOp Incidencias Operación Propuestas Sicsep2003 sui2002 V2 VR,Windows,No Aplica,Windows Server 2000,SI,"Registra los accesos al aplicativo (tablas ACCESOS, ACCESOS04, ACCESOS2)(Usuario, Fecha-hora, IP, Módulo)",No Aplica,NO,Aplicación,SI,Autenticación Propia,2026-10-20T22:48:09.176Z
Declaraciones y Pagos (Informativas y Pagos Provisionales),Pagos Electrónicos y Declaraciones,APE1,Ignacio Ramón Ruíz Casillas,AGR,12,12,Consulta de Declaraciones y Pagos Web,CUN-WEB,Consulta Integral de todas las Declaraciones y Pagos.,"La CUNWEB reúne en una misma aplicación la información de los pagos, declaraciones y datos de registro de los contribuyentes.",Ricardo Tejeda de la Cruz,Yeny Prudente Morales,Elizabeth del Valle Beltran,Fábrica SDMA 7 DS,null,Alta,3,Crítica,Legada,Asp Visual Basic 6.0 XML .NET VS 2010 framework 4.0 Informix 11.11,Microsoft SQL Server 2008 R2 ENTERPRISE Service Pack 3,DBCSEGCU transaccion transacciondm,MICROSOFT WINDOWS 2000 ADVANCED SERVER,No Aplica,Windows Server 2008 HP-UX B.11.31,SI,"Idsession, Usuario, RFC, FechaConsulta, Modulo, FechaAccesso",No Aplica,NO,Aplicación,SI,Autenticación Propia,2026-10-20T22:48:09.176Z
Fiscalización,Ejecutar Actos de Fiscalización,APE2,Alejandro Federico Tejeda Muñoz,AGAFF,17,17,Control y Registro de Usuarios,CORUS (IDC),"Sistema que controla la autorización y la autenticación de aplicativos; así como, contar con un repositorio de catálogos de Administraciones.","Contar con un Sistema que permita registrar, controlar y monitorear usuarios; así como, contar con un registro de Sistemas y la relación entre ambos.",Raúl Quiroz Sosa,Ingrid Bernal Campuzano,Santa Isabel Villanueva Hernández,Fábrica SDMA 7 DS,null,Alta,2,Crítica,Legada,.NET,Microsoft SQL Server 2008 R2 Enterprise Service Pack 3,CORUS,Windows Server 2012,Information Server Biztalk Server,SQL Server 2012,SI,Guarda cuando un usuario intenta acceder a u